# SINAL LIBRAS — auditoria do dataset UECE

Este notebook usa o ambiente gratuito do Google Colab para baixar e inspecionar o pacote de landmarks (aprox. 774 MB). Não treina modelo, não envia dados pessoais e não modifica o dataset original. O uso do corpus deve respeitar a licença CC BY-NC 4.0 e a atribuição aos autores.

**Atenção:** o download pode demorar e consumir cerca de 774 MB de tráfego e espaço temporário.

In [ ]:
from pathlib import Path
import hashlib, json, requests, subprocess, sys, os, time

RECORD_ID = '21398895'
EXPECTED_MD5 = '415a18d9be6d65468bcf92db5e940df8'
WORK = Path('/content/sinal_libras_uece')
WORK.mkdir(exist_ok=True)
print('Ambiente preparado:', WORK)

In [ ]:
# Consulta a API oficial do Zenodo e localiza exatamente Landmarks.zip
api = requests.get(f'https://zenodo.org/api/records/{RECORD_ID}', timeout=60)
api.raise_for_status()
record = api.json()
matches = [f for f in record['files'] if f['key'].lower() == 'landmarks.zip']
if not matches:
    raise RuntimeError('Landmarks.zip não encontrado no registro; confira a versão do dataset.')
file_info = matches[0]
print('Arquivo:', file_info['key'])
print('Tamanho (MB):', round(file_info['size']/1024**2, 1))
print('Checksum:', file_info['checksum'])
print('Link:', file_info['links']['self'])
if file_info['checksum'].split(':')[-1] != EXPECTED_MD5:
    raise RuntimeError('O checksum publicado mudou. Pare e revise o registro antes de baixar.')

In [ ]:
# Download em blocos, com verificação MD5 ao final
archive = WORK / 'Landmarks.zip'
if not archive.exists() or archive.stat().st_size != file_info['size']:
    h = hashlib.md5()
    total = 0
    with requests.get(file_info['links']['self'], stream=True, timeout=(30, 180)) as response:
        response.raise_for_status()
        with archive.open('wb') as out:
            for chunk in response.iter_content(chunk_size=8*1024*1024):
                if chunk:
                    out.write(chunk); h.update(chunk); total += len(chunk)
                    print(f'Baixados: {total/1024**2:.0f} MB / {file_info["size"]/1024**2:.0f} MB', end='\\r')
    print('\\nMD5 calculado:', h.hexdigest())
else:
    print('Arquivo já existe com tamanho esperado; calculando MD5...')
    h = hashlib.md5()
    with archive.open('rb') as src:
        for chunk in iter(lambda: src.read(8*1024*1024), b''):
            h.update(chunk)
    print('MD5 calculado:', h.hexdigest())
if archive.stat().st_size != file_info['size'] or h.hexdigest() != EXPECTED_MD5:
    raise RuntimeError('Download incompleto ou checksum incorreto; não use esses dados.')
print('Download verificado:', archive)

In [ ]:
# Baixa somente o inspetor do repositório e lista os formatos/cabeçalhos sem extrair o ZIP
import urllib.request
inspector_url = 'https://raw.githubusercontent.com/danyellallk/SINAL-LIBRAS/main/training/inspect_libras_eqt.py'
inspector_path = WORK / 'inspect_libras_eqt.py'
urllib.request.urlretrieve(inspector_url, inspector_path)
result = subprocess.run([sys.executable, str(inspector_path), str(archive), '--limit', '40'], text=True, capture_output=True)
print(result.stdout)
if result.stderr: print('AVISOS/ERROS:', result.stderr)
if result.returncode != 0: raise RuntimeError('O inspetor encontrou erro; copie a saída acima para análise.')

## Próxima etapa
Copie a saída completa da inspeção (nomes de arquivos, cabeçalhos CSV/JSON e metadados NPY) e envie aqui. Com o formato real confirmado, será possível escrever um conversor correto para o JSONL de treinamento. **Não execute treinamento ainda e não habilite `allow_pickle=True` em arquivos desconhecidos.**